In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [20]:
pip install kagglehub[pandas-datasets]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import sqlite3
import pandas as pd
import os

# Create a directory to save CSV files
output_dir = "f1_data_csv"
os.makedirs(output_dir, exist_ok=True)

# Connect to your local SQLite database
db_file = "Formula1.sqlite"  

print(f"Connecting to {db_file}...")

try:
    conn = sqlite3.connect(db_file)
    

    tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
    tables_df = pd.read_sql_query(tables_query, conn)
    
    print("\nTables in the database:")
    table_names = tables_df['name'].tolist()
    for table in table_names:
        print(f" - {table}")
    

    for table_name in table_names:
        try:
            print(f"\nExporting {table_name} to CSV...")
            
 
            df = pd.read_sql_query(f"SELECT * FROM {table_name};", conn)
            
   
            csv_filename = f"{output_dir}/{table_name}.csv"
            df.to_csv(csv_filename, index=False)
            print(f"✓ Saved {len(df)} records to {csv_filename}")
            
        except Exception as e:
            print(f"✗ Error exporting {table_name}: {e}")
    

    conn.close()
    
    print(f"\nAll CSV files have been saved to the '{output_dir}' directory!")
    
    # Listing all created CSV files
    print("\nCreated CSV files:")
    for file in sorted(os.listdir(output_dir)):
        if file.endswith('.csv'):
            file_size = os.path.getsize(f"{output_dir}/{file}")
            print(f" - {file} ({file_size:,} bytes)")
            
except FileNotFoundError:
    print(f"Error: Could not find {db_file} in the current directory")
except Exception as e:
    print(f"Error: {e}")

Connecting to Formula1.sqlite...

Tables in the database:
 - circuits
 - races
 - driver_standings
 - drivers
 - constructors
 - results
 - constructor_standings
 - constructor_results
 - laptimes
 - pitstops
 - qualifying
 - seasons
 - status

Exporting circuits to CSV...
✓ Saved 73 records to f1_data_csv/circuits.csv

Exporting races to CSV...
✓ Saved 997 records to f1_data_csv/races.csv

Exporting driver_standings to CSV...
✓ Saved 31726 records to f1_data_csv/driver_standings.csv

Exporting drivers to CSV...
✓ Saved 842 records to f1_data_csv/drivers.csv

Exporting constructors to CSV...
✓ Saved 208 records to f1_data_csv/constructors.csv

Exporting results to CSV...
✓ Saved 23777 records to f1_data_csv/results.csv

Exporting constructor_standings to CSV...
✓ Saved 11896 records to f1_data_csv/constructor_standings.csv

Exporting constructor_results to CSV...
✓ Saved 11142 records to f1_data_csv/constructor_results.csv

Exporting laptimes to CSV...
✓ Saved 426633 records to f1_data

In [23]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
import sqlite3


input_dir = "f1_data_csv"
output_dir = "f1_data_cleaned"
sqlite_file = "Formula1.sqlite"  
os.makedirs(output_dir, exist_ok=True)

print("=" * 80)
print("FORMULA 1 DATA CLEANING WITH RELATIONSHIP VALIDATION")
print("=" * 80)

# Defining the relationships 
RELATIONSHIPS = {
    'races': {
        'foreign_keys': {
            'circuitId': ('circuits', 'circuitId')
        }
    },
    'results': {
        'foreign_keys': {
            'raceId': ('races', 'raceId'),
            'driverId': ('drivers', 'driverId'),
            'constructorId': ('constructors', 'constructorId'),
            'statusId': ('status', 'statusId')
        }
    },
    'qualifying': {
        'foreign_keys': {
            'raceId': ('races', 'raceId'),
            'driverId': ('drivers', 'driverId'),
            'constructorId': ('constructors', 'constructorId')
        }
    },
    'lap_times': {
        'foreign_keys': {
            'raceId': ('races', 'raceId'),
            'driverId': ('drivers', 'driverId')
        }
    },
    'pit_stops': {
        'foreign_keys': {
            'raceId': ('races', 'raceId'),
            'driverId': ('drivers', 'driverId')
        }
    },
    'driver_standings': {
        'foreign_keys': {
            'raceId': ('races', 'raceId'),
            'driverId': ('drivers', 'driverId')
        }
    },
    'constructor_standings': {
        'foreign_keys': {
            'raceId': ('races', 'raceId'),
            'constructorId': ('constructors', 'constructorId')
        }
    },
    'constructor_results': {
        'foreign_keys': {
            'raceId': ('races', 'raceId'),
            'constructorId': ('constructors', 'constructorId')
        }
    }
}

# Loading all tables
tables = {}
print("\n📂 Loading all tables...")
for csv_file in sorted(os.listdir(input_dir)):
    if csv_file.endswith('.csv'):
        table_name = csv_file.replace('.csv', '')
        tables[table_name] = pd.read_csv(f"{input_dir}/{csv_file}")
        print(f"  ✓ {table_name}: {len(tables[table_name])} rows")

# Data quality report
quality_report = {
    'tables': {},
    'relationships': {},
    'integrity_issues': []
}

def validate_foreign_keys(table_name, df, fk_config):
    """Validate foreign key relationships"""
    issues = []
    
    for fk_col, (ref_table, ref_col) in fk_config.items():
        if fk_col not in df.columns:
            continue
            
        if ref_table not in tables:
            issues.append(f"Referenced table '{ref_table}' not found")
            continue
        
        # Get foreign key values (excluding nulls)
        fk_values = df[fk_col].dropna().unique()
        ref_values = tables[ref_table][ref_col].unique()
        
        # Find orphaned records (FK values that don't exist in referenced table)
        orphaned = set(fk_values) - set(ref_values)
        
        if orphaned:
            issues.append({
                'column': fk_col,
                'references': f"{ref_table}.{ref_col}",
                'orphaned_count': len(orphaned),
                'orphaned_values': list(orphaned)[:10]  
            })
    
    return issues

def clean_table_advanced(table_name, df):
    """Advanced cleaning with relationship awareness"""
    print(f"\n{'='*80}")
    print(f"🔧 Processing: {table_name}")
    print(f"{'='*80}")
    print(f"Original shape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    original_rows = len(df)
    report = {
        'original_rows': original_rows,
        'missing_values': {},
        'data_types': {},
        'cleaned_rows': 0,
        'issues_found': [],
        'foreign_key_issues': []
    }
    
    # 1. Check missing values
    print("\n Missing Values Analysis:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    has_missing = False
    for col in df.columns:
        if missing[col] > 0:
            if not has_missing:
                has_missing = True
            print(f"   {col}: {missing[col]} ({missing_pct[col]}%)")
            report['missing_values'][col] = {
                'count': int(missing[col]),
                'percentage': float(missing_pct[col])
            }
    
    if not has_missing:
        print("  No missing values found")
    
    # 2. Validate foreign key relationships
    if table_name in RELATIONSHIPS and 'foreign_keys' in RELATIONSHIPS[table_name]:
        print(f"\n Validating Foreign Key Relationships:")
        fk_issues = validate_foreign_keys(
            table_name, 
            df, 
            RELATIONSHIPS[table_name]['foreign_keys']
        )
        
        if fk_issues:
            report['foreign_key_issues'] = fk_issues
            for issue in fk_issues:
                if isinstance(issue, dict):
                    print(f"   {issue['column']} -> {issue['references']}")
                    print(f"      Orphaned records: {issue['orphaned_count']}")
                    if issue['orphaned_count'] <= 10:
                        print(f"      Values: {issue['orphaned_values']}")
                else:
                    print(f"   {issue}")
        else:
            print("  All foreign key relationships valid")
    

    df_cleaned = df.copy()
    
    if table_name == 'drivers':
        # Cleaning driver data
        if 'dob' in df_cleaned.columns:
            df_cleaned['dob'] = pd.to_datetime(df_cleaned['dob'], errors='coerce')
            invalid_dob = df_cleaned['dob'].isnull().sum() - missing['dob']
            if invalid_dob > 0:
                report['issues_found'].append(f"Corrected {invalid_dob} invalid birth dates")
        
        # Driver number can be NULL for older drivers (pre-2014)
        if 'number' in df_cleaned.columns:
            null_numbers = df_cleaned['number'].isnull().sum()
            if null_numbers > 0:
                report['issues_found'].append(f"{null_numbers} drivers without numbers (pre-2014 era - expected)")
    
    elif table_name == 'races':
        # Clean race data
        if 'date' in df_cleaned.columns:
            df_cleaned['date'] = pd.to_datetime(df_cleaned['date'], errors='coerce')
            invalid_dates = df_cleaned['date'].isnull().sum() - missing['date']
            if invalid_dates > 0:
                report['issues_found'].append(f"Corrected {invalid_dates} invalid dates")
        
        # Time can be NULL for older races
        if 'time' in df_cleaned.columns:
            null_times = df_cleaned['time'].isnull().sum()
            if null_times > 0:
                report['issues_found'].append(f"{null_times} races without time data (older races)")
        
        # Validate year matches season
        if 'year' in df_cleaned.columns:
            year_range = f"{df_cleaned['year'].min()}-{df_cleaned['year'].max()}"
            print(f" Year range: {year_range}")
    
    elif table_name == 'results':
        # Results table is central - lots of NULLs are expected for DNF/DSQ
        timing_cols = ['time', 'milliseconds', 'fastestLap', 'fastestLapTime', 'fastestLapSpeed']
        for col in timing_cols:
            if col in df_cleaned.columns:
                null_count = df_cleaned[col].isnull().sum()
                if null_count > 0:
                    report['issues_found'].append(f"{null_count} results missing {col} (DNF/DSQ - expected)")
        
        # Position can be NULL or text (R, D, E, W, F, N for DNF statuses)
        if 'position' in df_cleaned.columns and 'positionText' in df_cleaned.columns:
            # Keep positionText as-is, convert position to numeric
            df_cleaned['position'] = pd.to_numeric(df_cleaned['position'], errors='coerce')
        
        # Points should be numeric
        if 'points' in df_cleaned.columns:
            df_cleaned['points'] = pd.to_numeric(df_cleaned['points'], errors='coerce')
    
    elif table_name == 'qualifying':
        # Q2 and Q3 are NULL if driver didn't advance
        for q_round in ['q2', 'q3']:
            if q_round in df_cleaned.columns:
                null_count = df_cleaned[q_round].isnull().sum()
                if null_count > 0:
                    report['issues_found'].append(f"{null_count} drivers didn't reach {q_round.upper()} (expected)")
    
    elif table_name == 'pit_stops':
        # Clean pit stop durations
        if 'duration' in df_cleaned.columns:
            df_cleaned['duration'] = pd.to_numeric(df_cleaned['duration'], errors='coerce')
        if 'milliseconds' in df_cleaned.columns:
            df_cleaned['milliseconds'] = pd.to_numeric(df_cleaned['milliseconds'], errors='coerce')
    
    elif table_name == 'circuits':
        # Altitude can be NULL
        if 'alt' in df_cleaned.columns:
            null_alt = df_cleaned['alt'].isnull().sum()
            if null_alt > 0:
                report['issues_found'].append(f"{null_alt} circuits missing altitude data")
        
        # Lat/Lng should be numeric
        for col in ['lat', 'lng']:
            if col in df_cleaned.columns:
                df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
    
    elif table_name == 'constructors':
        # Ensure no duplicates on constructorId
        if 'constructorId' in df_cleaned.columns:
            dupes = df_cleaned['constructorId'].duplicated().sum()
            if dupes > 0:
                print(f"   Found {dupes} duplicate constructorIds")
                df_cleaned = df_cleaned.drop_duplicates(subset=['constructorId'], keep='first')
                report['issues_found'].append(f"Removed {dupes} duplicate constructors")
    
    elif table_name == 'status':
        # Status table should be clean
        if 'statusId' in df_cleaned.columns:
            dupes = df_cleaned['statusId'].duplicated().sum()
            if dupes > 0:
                df_cleaned = df_cleaned.drop_duplicates(subset=['statusId'], keep='first')
                report['issues_found'].append(f"Removed {dupes} duplicate status records")
    
    columns_to_drop = {
        'constructor_standings': ['positionText'],
        'driver_standings': ['positionText'],
        'results': ['positionText', 'positionOrder']
    }

    if table_name in columns_to_drop:
        for col in columns_to_drop[table_name]:
            if col in df_cleaned.columns:
                df_cleaned = df_cleaned.drop(columns=[col])
                print(f"🧹 Dropped column: {col} from {table_name}")

    # 4. Remove completely empty rows
    df_cleaned = df_cleaned.dropna(how='all')
    rows_removed = original_rows - len(df_cleaned)
    if rows_removed > 0:
        print(f"\n  🗑️  Removed {rows_removed} completely empty rows")
        report['issues_found'].append(f"Removed {rows_removed} empty rows")
    
    # 5. Data type summary
    print(f"\n Data Types Summary:")
    for col in df_cleaned.columns[:5]: 
        dtype = str(df_cleaned[col].dtype)
        print(f"  • {col}: {dtype}")
        report['data_types'][col] = dtype
    if len(df_cleaned.columns) > 5:
        print(f"  ... and {len(df_cleaned.columns) - 5} more columns")
    
    report['cleaned_rows'] = len(df_cleaned)
    
    print(f"\nCleaning complete: {df_cleaned.shape[0]} rows, {df_cleaned.shape[1]} columns")
    
    return df_cleaned, report

# Process all tables
print("\n" + "="*80)
print(" Starting table-by-table cleaning...")
print("="*80)

for table_name in sorted(tables.keys()):
    try:
        df_cleaned, report = clean_table_advanced(table_name, tables[table_name])
        
        # Save cleaned data
        output_file = f"{output_dir}/{table_name}.csv"
        df_cleaned.to_csv(output_file, index=False)
        print(f"💾 Saved to: {output_file}")
        
        # Update tables dict with cleaned version
        tables[table_name] = df_cleaned
        
        # Store report
        quality_report['tables'][table_name] = report
        
    except Exception as e:
        print(f"\n Error processing {table_name}: {e}")
        import traceback
        traceback.print_exc()


print("\n" + "="*80)
print("📊 COMPREHENSIVE DATA QUALITY REPORT")
print("="*80)

total_original = sum(r['original_rows'] for r in quality_report['tables'].values())
total_cleaned = sum(r['cleaned_rows'] for r in quality_report['tables'].values())

print(f"\n Overall Statistics:")
print(f"  • Total tables: {len(quality_report['tables'])}")
print(f"  • Total rows (original): {total_original:,}")
print(f"  • Total rows (cleaned): {total_cleaned:,}")
print(f"  • Rows removed: {total_original - total_cleaned:,}")
print(f"  • Data quality: {(total_cleaned/total_original*100):.2f}%")


print(f"\n🔴 Critical Issues (Foreign Key Violations):")
critical_count = 0
for table, report in quality_report['tables'].items():
    if report['foreign_key_issues']:
        critical_count += 1
        print(f"\n  {table}:")
        for issue in report['foreign_key_issues']:
            if isinstance(issue, dict):
                print(f"     {issue['column']} -> {issue['references']}: {issue['orphaned_count']} orphaned records")

if critical_count == 0:
    print("  No critical foreign key violations found!")


print(f"\n  Tables with Missing Data (>5%):")
missing_count = 0
for table, report in quality_report['tables'].items():
    significant_missing = {k: v for k, v in report['missing_values'].items() if v['percentage'] > 5}
    if significant_missing:
        missing_count += 1
        print(f"\n  {table}:")
        for col, info in significant_missing.items():
            print(f"    • {col}: {info['count']} ({info['percentage']}%)")

if missing_count == 0:
    print("  No significant missing data issues!")


print(f"\n Expected Data Quality Notes:")
expected_notes = [
    "• Driver numbers NULL for pre-2014 drivers (expected)",
    "• Race times NULL for older races (expected)",
    "• Result timing data NULL for DNF/DSQ drivers (expected)",
    "• Qualifying Q2/Q3 times NULL for eliminated drivers (expected)",
    "• Circuit altitude data may be missing for some tracks"
]
for note in expected_notes:
    print(note)

print("\n" + "="*80)
print(" DATA CLEANING COMPLETE - READY FOR POSTGRESQL!")
print("="*80)
print(f"\n Cleaned files location: {output_dir}/")
print(f" Next step: Load into PostgreSQL using ETL script")
print("="*80)


import json
with open('data_quality_report_detailed.json', 'w') as f:
    def convert(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    
    report_serializable = json.loads(json.dumps(quality_report, default=convert))
    json.dump(report_serializable, f, indent=2)

print(f"\n Detailed report saved: data_quality_report_detailed.json")

FORMULA 1 DATA CLEANING WITH RELATIONSHIP VALIDATION

📂 Loading all tables...
  ✓ circuits: 73 rows
  ✓ constructor_results: 11142 rows
  ✓ constructor_standings: 11896 rows
  ✓ constructors: 208 rows
  ✓ driver_standings: 31726 rows
  ✓ drivers: 842 rows
  ✓ laptimes: 426633 rows
  ✓ pitstops: 6251 rows
  ✓ qualifying: 7516 rows
  ✓ races: 997 rows
  ✓ results: 23777 rows
  ✓ seasons: 69 rows
  ✓ status: 134 rows

 Starting table-by-table cleaning...

🔧 Processing: circuits
Original shape: 73 rows, 9 columns

 Missing Values Analysis:
   alt: 72 (98.63%)

 Data Types Summary:
  • circuitId: int64
  • circuitRef: object
  • name: object
  • location: object
  • country: object
  ... and 4 more columns

Cleaning complete: 73 rows, 9 columns
💾 Saved to: f1_data_cleaned/circuits.csv

🔧 Processing: constructor_results
Original shape: 11142 rows, 5 columns

 Missing Values Analysis:
   status: 11125 (99.85%)

 Validating Foreign Key Relationships:
  All foreign key relationships valid

 Dat

In [18]:
import pandas as pd
import os
import sqlite3
import numpy as np
from datetime import datetime


input_dir = "f1_data_cleaned"  
output_dir = "f1_data_postgres_ready"
os.makedirs(output_dir, exist_ok=True)

print("=" * 80)
print("FINAL POSTGRESQL PREPARATION - FIXING REMAINING ISSUES")
print("=" * 80)


FIXES_NEEDED = {
    'constructor_standings': {
        'remove_columns': ['Unnamed: 7']
    },
    'constructors': {
        'remove_columns': ['Unnamed: 5']
    },
    'drivers': {
        'fix_dob': True 
    }
}

def fix_table(table_name, df):
    """Apply final fixes to each table"""
    print(f"\n{'='*80}")
    print(f"🔧 Final cleanup: {table_name}")
    print(f"{'='*80}")
    print(f"Input shape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    df_fixed = df.copy()
    fixes_applied = []
    

    if table_name in FIXES_NEEDED and 'remove_columns' in FIXES_NEEDED[table_name]:
        cols_to_remove = FIXES_NEEDED[table_name]['remove_columns']
        existing_cols = [col for col in cols_to_remove if col in df_fixed.columns]
        
        if existing_cols:
            df_fixed = df_fixed.drop(columns=existing_cols)
            fixes_applied.append(f"Removed columns: {', '.join(existing_cols)}")
            print(f"  ✅ Removed {len(existing_cols)} unnamed column(s)")
    
    # Fixing specific table issues
    if table_name == 'drivers' and 'dob' in df_fixed.columns:
        # Re-parse DOB more carefully
        print(f"  🔍 Analyzing birth dates...")
        
        # Try to parse with multiple date formats
        def parse_dob(date_str):
            if pd.isna(date_str):
                return None
            
            # Common date formats
            formats = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d']
            
            for fmt in formats:
                try:
                    return pd.to_datetime(date_str, format=fmt)
                except:
                    continue
            
            #error handling
            try:
                parsed = pd.to_datetime(date_str, errors='coerce')
                if parsed and 1900 <= parsed.year <= 2010:
                    return parsed
            except:
                pass
            
            return None
        
        original_nulls = df_fixed['dob'].isna().sum()
        
        # Apply parsing
        df_fixed['dob'] = df_fixed['dob'].apply(parse_dob)
        
        final_nulls = df_fixed['dob'].isna().sum()
        fixed_count = final_nulls - original_nulls
        
        if fixed_count > 0:
            print(f"  ⚠️  Could not parse {fixed_count} birth dates - set to NULL")
            fixes_applied.append(f"Fixed DOB parsing: {fixed_count} unparseable dates set to NULL")
        else:
            print(f"  ✅ All birth dates valid")
    
    # Ensuring proper data types for PostgreSQL
    print(f"\n  🔢 Validating data types...")
    
    # Integer columns should not have floats unless needed
    for col in df_fixed.columns:
        if df_fixed[col].dtype == 'float64':
            # Check if this should be integer
            if col.lower().endswith('id') or col in ['year', 'round', 'number', 'lap', 'position', 'stop', 'grid']:
                try:
                    if df_fixed[col].notna().all() or (df_fixed[col].dropna() == df_fixed[col].dropna().astype(int)).all():
                        df_fixed[col] = df_fixed[col].astype('Int64')
                        fixes_applied.append(f"Converted {col} to Int64 (nullable integer)")
                except:
                    pass
    
    # Handling NULLs appropriately for PostgreSQL
    # Replace pandas NA/NaN with proper NULL handling
    df_fixed = df_fixed.replace({np.nan: None, pd.NaT: None})
    
    # Validate column names for PostgreSQL
    # PostgreSQL prefers lowercase with underscores
    original_columns = df_fixed.columns.tolist()
    new_columns = []
    
    for col in original_columns:
        new_col = col
        
        # Check for problematic characters
        if any(char in col for char in [' ', '-', '.']):
            new_col = col.replace(' ', '_').replace('-', '_').replace('.', '_')
            fixes_applied.append(f"Renamed column: {col} -> {new_col}")
        
        new_columns.append(new_col)
    
    df_fixed.columns = new_columns
    
    # 6. Final validation
    print(f"\n Final statistics:")
    print(f"     • Rows: {len(df_fixed)}")
    print(f"     • Columns: {len(df_fixed.columns)}")
    print(f"     • Total NULLs: {df_fixed.isna().sum().sum()}")
    print(f"     • Memory usage: {df_fixed.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    
    if fixes_applied:
        print(f"\n  Fixes applied:")
        for fix in fixes_applied:
            print(f"     • {fix}")
    else:
        print(f"\n  No fixes needed - table is PostgreSQL-ready")
    
    print(f"\n Final shape: {df_fixed.shape[0]} rows, {df_fixed.shape[1]} columns")
    
    return df_fixed, fixes_applied


print("\nProcessing all tables...")

summary = {
    'tables_processed': 0,
    'total_fixes': 0,
    'tables': {}
}

for csv_file in sorted(os.listdir(input_dir)):
    if csv_file.endswith('.csv'):
        table_name = csv_file.replace('.csv', '')
        
        try:
            # Load table
            df = pd.read_csv(f"{input_dir}/{csv_file}")
            
            # Apply fixes
            df_fixed, fixes = fix_table(table_name, df)
            
            # Save PostgreSQL-ready version
            output_file = f"{output_dir}/{csv_file}"
            df_fixed.to_csv(output_file, index=False, na_rep='\\N')  # PostgreSQL NULL representation
            print(f"💾 Saved to: {output_file}")
            
            # Update summary
            summary['tables_processed'] += 1
            summary['total_fixes'] += len(fixes)
            summary['tables'][table_name] = {
                'rows': len(df_fixed),
                'columns': len(df_fixed.columns),
                'fixes_applied': fixes
            }
            
        except Exception as e:
            print(f"\nError processing {table_name}: {e}")
            import traceback
            traceback.print_exc()

# Generate final report
print("\n" + "="*80)
print("FINAL POSTGRESQL PREPARATION SUMMARY")
print("="*80)

print(f"\nTables processed: {summary['tables_processed']}")
print(f"Total fixes applied: {summary['total_fixes']}")

print(f"\nTable Summary:")
for table_name, info in sorted(summary['tables'].items()):
    print(f"\n  {table_name}:")
    print(f"    • Rows: {info['rows']:,}")
    print(f"    • Columns: {info['columns']}")
    if info['fixes_applied']:
        print(f"    • Fixes: {len(info['fixes_applied'])}")
        for fix in info['fixes_applied']:
            print(f"      - {fix}")
    else:
        print(f"    • No fixes needed ✅")

# Create PostgreSQL load order based on dependencies
print(f"\n" + "="*80)
print("RECOMMENDED POSTGRESQL LOAD ORDER")
print("="*80)
print("\nBased on foreign key dependencies, load tables in this order:\n")

load_order = [
    ("1. Master Tables (no dependencies)", [
        "seasons",
        "status", 
        "circuits",
        "drivers",
        "constructors"
    ]),
    ("2. Race Tables", [
        "races"
    ]),
    ("3. Race-Dependent Tables", [
        "results",
        "qualifying",
        "lap_times",
        "pit_stops",
        "driver_standings",
        "constructor_standings",
        "constructor_results"
    ])
]

for category, tables in load_order:
    print(f"{category}:")
    for table in tables:
        if table in summary['tables']:
            print(f"  ✓ {table} ({summary['tables'][table]['rows']:,} rows)")
        else:
            # Handle table name variations
            alt_name = table.replace('_', '')
            if alt_name in summary['tables']:
                print(f"  {alt_name} ({summary['tables'][alt_name]['rows']:,} rows)")
    print()

print("="*80)
print("DATA IS FULLY READY FOR POSTGRESQL!")
print("="*80)
print(f"\nPostgreSQL-ready files: {output_dir}/")
print("="*80)

# Save the summary report
import json
with open('postgres_preparation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n Summary saved: postgres_preparation_summary.json")

FINAL POSTGRESQL PREPARATION - FIXING REMAINING ISSUES

Processing all tables...

🔧 Final cleanup: circuits
Input shape: 73 rows, 9 columns

  🔢 Validating data types...

 Final statistics:
     • Rows: 73
     • Columns: 9
     • Total NULLs: 72
     • Memory usage: 0.03 MB

  No fixes needed - table is PostgreSQL-ready

 Final shape: 73 rows, 9 columns
💾 Saved to: f1_data_postgres_ready/circuits.csv

🔧 Final cleanup: constructor_results
Input shape: 11142 rows, 5 columns

  🔢 Validating data types...

 Final statistics:
     • Rows: 11142
     • Columns: 5
     • Total NULLs: 11125
     • Memory usage: 0.60 MB

  No fixes needed - table is PostgreSQL-ready

 Final shape: 11142 rows, 5 columns
💾 Saved to: f1_data_postgres_ready/constructor_results.csv

🔧 Final cleanup: constructor_standings
Input shape: 11896 rows, 8 columns
  ✅ Removed 1 unnamed column(s)

  🔢 Validating data types...

 Final statistics:
     • Rows: 11896
     • Columns: 7
     • Total NULLs: 0
     • Memory usage: 

In [13]:
pip install psycopg2-binary


   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
    --------------------------------------- 0.0/2.7 MB 653.6 kB/s eta 0:00:05
   --- ------------------------------------ 0.3/2.7 MB 2.2 MB/s eta 0:00:02
   ----------- ---------------------------- 0.8/2.7 MB 5.1 MB/s eta 0:00:01
   -------------------- ------------------- 1.4/2.7 MB 6.2 MB/s eta 0:00:01
   ----------------------------- ---------- 2.0/2.7 MB 7.4 MB/s eta 0:00:01
   ------------------------------------ --- 2.5/2.7 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 7.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
